<a href="https://colab.research.google.com/github/barrettparis/barrettparis/blob/circleci-project-setup/Web_ADB_Server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import http.server
import socketserver
import json
import subprocess
import os
import cgi

PORT = 8000
GUI_FILE = "web-adb-gui.html"

class ADBRequestHandler(http.server.SimpleHTTPRequestHandler):

    def _send_json(self, data, status=200):
        """Helper to send JSON responses."""
        self.send_response(status)
        self.send_header("Content-type", "application/json")
        self.end_headers()
        self.wfile.write(json.dumps(data).encode("utf-8"))

    def _run_command(self, command_list):
        """Runs a command and returns its output or error."""
        try:
            # We use a list of args for security, not a single string
            result = subprocess.run(
                command_list,
                capture_output=True,
                text=True,
                check=True,
                encoding="utf-8"
            )
            return {"success": True, "output": result.stdout.strip()}
        except FileNotFoundError:
            return {"success": False, "error": f"Error: 'adb' command not found. Is it in your system PATH?"}
        except subprocess.CalledProcessError as e:
            return {"success": False, "error": e.stderr.strip() or e.stdout.strip()}
        except Exception as e:
            return {"success": False, "error": f"A server error occurred: {str(e)}"}

    def do_GET(self):
        """Handles GET requests. Serves the HTML GUI."""
        if self.path == "/":
            self.path = GUI_FILE
            return http.server.SimpleHTTPRequestHandler.do_GET(self)
        else:
            # Failsafe to prevent directory traversal
            if ".." in self.path:
                self.send_error(403, "Forbidden")
                return

            # Check if requested file exists, otherwise serve the GUI
            # This helps for relative paths if needed
            safe_path = os.path.join(os.getcwd(), self.path.lstrip('/'))
            if not os.path.exists(safe_path):
                 self.path = GUI_FILE
                 return http.server.SimpleHTTPRequestHandler.do_GET(self)

            return http.server.SimpleHTTPRequestHandler.do_GET(self)

    def do_POST(self):
        """Handles POST requests from the GUI (all our API calls)."""
        content_length = int(self.headers.get("Content-Length", 0))
        post_data = self.rfile.read(content_length)

        try:
            body = json.loads(post_data.decode("utf-8"))
        except json.JSONDecodeError:
            return self._send_json({"success": False, "error": "Invalid JSON"}, 400)

        endpoint = self.path
        print(f"[Server] Received POST to {endpoint}: {body}")

        if endpoint == "/api/run-command":
            command = body.get("command", [])
            if not command:
                return self._send_json({"success": False, "error": "No command provided"}, 400)

            # We prepend 'adb' to the command list
            result = self._run_command(["adb"] + command)
            return self._send_json(result)

        elif endpoint == "/api/install-apk":
            # This is more complex in a real app (would need file upload)
            # For now, we just pass the *local path* from the user's computer
            path = body.get("path")
            if not path:
                return self._send_json({"success": False, "error": "No path provided"}, 400)

            # Basic sanitization
            if ".." in path or not (path.startswith("/") or path[1:3] == ":\\"):
                 if not os.path.exists(path):
                     return self._send_json({"success": False, "error": f"File not found on server: {path}"}, 404)

            result = self._run_command(["adb", "install", "-r", path])
            return self._send_json(result)

        return self._send_json({"success": False, "error": "Unknown endpoint"}, 404)

Handler = ADBRequestHandler

with socketserver.TCPServer(("", PORT), Handler) as httpd:
    print(f"Serving Web ADB GUI at http://localhost:{PORT}")
    print("Press Ctrl+C to stop the server.")
    try:
        httpd.serve_forever()
    except KeyboardInterrupt:
        print("\nStopping server...")
        httpd.server_close()